In [2]:
import torch
import torch.nn as nn
import math

In [3]:

class InputEmbeddings(nn.Module):

    def __init__(self, d_model: int, vocab_size: int) -> None:
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        # (batch, seq_len) --> (batch, seq_len, d_model)
        # Multiply by sqrt(d_model) to scale the embeddings according to the paper
        return self.embedding(x) * math.sqrt(self.d_model)
    

In [24]:

# Set parameters
d_model = 4    # Embedding dimension
vocab_size = 10  # Vocabulary size

embedding_layer = InputEmbeddings(d_model, vocab_size)

# Sample input: a batch of 10 sequences, each of length 3
x = torch.tensor([[0.1, 0.3, 0.5], 
[0.2, 0.4, 0.6], 
[0.05, 0.2, 0.4], 
[0.25, 0.5, 0.7], 
[0.08, 0.22, 0.52], 
[0.3, 0.55, 0.75], 
[0.07, 0.21, 0.53], 
[0.25, 0.5, 0.8], 
[0.06, 0.2, 0.45], 
[0.24, 0.48, 0.85]], dtype=torch.long)

# Get the output embeddings
embeddings = embedding_layer(x)

# Display the vectors
print("Output embeddings:\n", embeddings)

Output embeddings:
 tensor([[[-1.3687, -0.9641, -4.0613, -4.2017],
         [-1.3687, -0.9641, -4.0613, -4.2017],
         [-1.3687, -0.9641, -4.0613, -4.2017]],

        [[-1.3687, -0.9641, -4.0613, -4.2017],
         [-1.3687, -0.9641, -4.0613, -4.2017],
         [-1.3687, -0.9641, -4.0613, -4.2017]],

        [[-1.3687, -0.9641, -4.0613, -4.2017],
         [-1.3687, -0.9641, -4.0613, -4.2017],
         [-1.3687, -0.9641, -4.0613, -4.2017]],

        [[-1.3687, -0.9641, -4.0613, -4.2017],
         [-1.3687, -0.9641, -4.0613, -4.2017],
         [-1.3687, -0.9641, -4.0613, -4.2017]],

        [[-1.3687, -0.9641, -4.0613, -4.2017],
         [-1.3687, -0.9641, -4.0613, -4.2017],
         [-1.3687, -0.9641, -4.0613, -4.2017]],

        [[-1.3687, -0.9641, -4.0613, -4.2017],
         [-1.3687, -0.9641, -4.0613, -4.2017],
         [-1.3687, -0.9641, -4.0613, -4.2017]],

        [[-1.3687, -0.9641, -4.0613, -4.2017],
         [-1.3687, -0.9641, -4.0613, -4.2017],
         [-1.3687, -0.9641, 

In [25]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model: int, seq_len: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)

        # Create a matrix of shape (seq_len, d_model)
        pe = torch.zeros(seq_len, d_model)
        # create a vector of shape (seq_len, 1)
        position = torch.arange(0, seq_len, dtype= torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float()* (-math.log(10000.0)/ d_model))

        #Apply the sin and cosin to even and odd positions respectivly

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False)
        return self.dropout(x)
    

In [26]:

pos_encoding = PositionalEncoding(d_model, seq_len=3, dropout=0.1)
output_pe = pos_encoding(embeddings)
print("Positional Encoding output:\n", output_pe)

Positional Encoding output:
 tensor([[[-1.5208,  0.0399, -0.0000, -3.5575],
         [-0.5859, -0.4708, -4.5014, -3.5576],
         [-0.5105, -1.5336, -4.4903, -3.5577]],

        [[-1.5208,  0.0399, -4.5125, -0.0000],
         [-0.5859, -0.4708, -4.5014, -3.5576],
         [-0.5105, -1.5336, -4.4903, -3.5577]],

        [[-1.5208,  0.0399, -4.5125, -3.5575],
         [-0.5859, -0.4708, -4.5014, -3.5576],
         [-0.0000, -1.5336, -4.4903, -3.5577]],

        [[-1.5208,  0.0399, -4.5125, -3.5575],
         [-0.5859, -0.4708, -4.5014, -3.5576],
         [-0.5105, -1.5336, -4.4903, -3.5577]],

        [[-1.5208,  0.0399, -4.5125, -3.5575],
         [-0.5859, -0.4708, -4.5014, -3.5576],
         [-0.5105, -0.0000, -4.4903, -3.5577]],

        [[-1.5208,  0.0399, -4.5125, -3.5575],
         [-0.5859, -0.4708, -4.5014, -0.0000],
         [-0.5105, -1.5336, -4.4903, -3.5577]],

        [[-1.5208,  0.0399, -4.5125, -0.0000],
         [-0.5859, -0.4708, -4.5014, -3.5576],
         [-0.5105, 

In [27]:

class MultiHeadAttentionBlock(nn.Module):

    def __init__(self, d_model: int, h: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model # Embedding vector size
        self.h = h # Number of heads
        # Make sure d_model is divisible by h
        assert d_model % h == 0, "d_model is not divisible by h"

        self.d_k = d_model // h # Dimension of vector seen by each head
        self.w_q = nn.Linear(d_model, d_model, bias=False) # Wq
        self.w_k = nn.Linear(d_model, d_model, bias=False) # Wk
        self.w_v = nn.Linear(d_model, d_model, bias=False) # Wv
        self.w_o = nn.Linear(d_model, d_model, bias=False) # Wo
        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.shape[-1]
        # Just apply the formula from the paper
        # (batch, h, seq_len, d_k) --> (batch, h, seq_len, seq_len)
        attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is not None:
            # Write a very low value (indicating -inf) to the positions where mask == 0
            attention_scores.masked_fill_(mask == 0, -1e9)
        attention_scores = attention_scores.softmax(dim=-1) # (batch, h, seq_len, seq_len) # Apply softmax
        if dropout is not None:
            attention_scores = dropout(attention_scores)
        # (batch, h, seq_len, seq_len) --> (batch, h, seq_len, d_k)
        # return attention scores which can be used for visualization
        return (attention_scores @ value), attention_scores

    def forward(self, q, k, v, mask):
        query = self.w_q(q) # (batch, seq_len, d_model) --> (batch, seq_len, d_model)
        key = self.w_k(k) # (batch, seq_len, d_model) --> (batch, seq_len, d_model)
        value = self.w_v(v) # (batch, seq_len, d_model) --> (batch, seq_len, d_model)

        # (batch, seq_len, d_model) --> (batch, seq_len, h, d_k) --> (batch, h, seq_len, d_k)
        query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1, 2)
        key = key.view(key.shape[0], key.shape[1], self.h, self.d_k).transpose(1, 2)
        value = value.view(value.shape[0], value.shape[1], self.h, self.d_k).transpose(1, 2)

        # Calculate attention
        x, self.attention_scores = MultiHeadAttentionBlock.attention(query, key, value, mask, self.dropout)
        
        # Combine all the heads together
        # (batch, h, seq_len, d_k) --> (batch, seq_len, h, d_k) --> (batch, seq_len, d_model)
        x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, self.h * self.d_k)

        # Multiply by Wo
        # (batch, seq_len, d_model) --> (batch, seq_len, d_model)  
        return self.w_o(x)

In [28]:
h = 2  # Number of attention heads
dropout = 0.1  # Dropout rate
multihead_attention = MultiHeadAttentionBlock(d_model, h, dropout)
mask = None
output_mha = multihead_attention(output_pe, output_pe, output_pe, mask)
print("Multi-head attention output:\n", output_mha)

Multi-head attention output:
 tensor([[[ 0.3043, -0.4827, -0.5545,  0.3869],
         [-0.2360, -0.0766, -0.2813,  0.2761],
         [-0.2258, -0.0889, -0.2857,  0.2779]],

        [[ 1.2387, -1.0319, -0.9154,  0.3065],
         [ 1.0454, -0.7046, -1.0385,  0.3000],
         [ 0.9207, -0.7893, -0.4548,  0.1975]],

        [[ 1.1245, -1.2579, -0.6589,  0.3197],
         [ 1.1586, -1.3309, -0.7978,  0.4447],
         [ 1.3572, -1.4960, -1.0006,  0.5835]],

        [[ 0.7373, -0.7405, -0.7591,  0.4698],
         [ 0.8839, -0.9953, -0.6886,  0.3978],
         [ 1.2801, -1.3815, -1.0312,  0.5754]],

        [[ 1.1015, -1.1415, -1.0995,  0.5661],
         [ 0.6235, -0.6321, -0.5739,  0.2481],
         [ 1.1015, -1.1424, -1.0983,  0.5658]],

        [[ 1.2110, -0.8776, -1.0041,  0.2997],
         [ 1.3952, -1.1727, -0.9756,  0.3541],
         [ 0.6196, -0.5300, -0.7785,  0.2404]],

        [[ 1.0929, -0.8096, -1.0026,  0.3075],
         [ 0.8828, -0.4878, -1.0445,  0.2578],
         [ 0.9126,

In [29]:
class LayerNormalization(nn.Module):
    def __init__(self, eps: float = 10**-6) -> None:
        super().__init__()
        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(1))
        self.bias = nn.Parameter(torch.ones(1))

    def forward(self, x):
        mean = x.mean(dim = -1, keepdim = True)
        std = x.std(dim = -1, keepdim = True) 
        return self.alpha * ((x-mean)/ (std + self.eps)) + self.bias

In [30]:
layer_norm = LayerNormalization(2)
output_ln = layer_norm(output_pe)
print("Layer Normalization output:\n", output_ln)

Layer Normalization output:
 tensor([[[0.9293, 1.3517, 1.3408, 0.3782],
         [1.4172, 1.4455, 0.4524, 0.6849],
         [1.5264, 1.2588, 0.4854, 0.7293]],

        [[0.9946, 1.3719, 0.2714, 1.3622],
         [1.4172, 1.4455, 0.4524, 0.6849],
         [1.5264, 1.2588, 0.4854, 0.7293]],

        [[1.2144, 1.6004, 0.4745, 0.7107],
         [1.4172, 1.4455, 0.4524, 0.6849],
         [1.5961, 1.2145, 0.4787, 0.7107]],

        [[1.2144, 1.6004, 0.4745, 0.7107],
         [1.4172, 1.4455, 0.4524, 0.6849],
         [1.5264, 1.2588, 0.4854, 0.7293]],

        [[1.2144, 1.6004, 0.4745, 0.7107],
         [1.4172, 1.4455, 0.4524, 0.6849],
         [1.3862, 1.5072, 0.4428, 0.6639]],

        [[1.2144, 1.6004, 0.4745, 0.7107],
         [1.1965, 1.2246, 0.2392, 1.3397],
         [1.5264, 1.2588, 0.4854, 0.7293]],

        [[0.9946, 1.3719, 0.2714, 1.3622],
         [1.4172, 1.4455, 0.4524, 0.6849],
         [1.3862, 1.5072, 0.4428, 0.6639]],

        [[0.9293, 1.3517, 1.3408, 0.3782],
         [1

In [31]:

class FeedForwardBlock(nn.Module):

    def __init__(self, d_model: int, d_ff: int, dropout: float) -> None:
        super().__init__()
        self.linear_1 = nn.Linear(d_model, d_ff) # w1 and b1
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model) # w2 and b2

    def forward(self, x):
        # (batch, seq_len, d_model) --> (batch, seq_len, d_ff) --> (batch, seq_len, d_model)
        return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))


In [32]:
d_ff = 8      
dropout = 0.1  

feed_forward_block = FeedForwardBlock(d_model, d_ff, dropout)
output_ff = feed_forward_block(output_ln)
print("Feedforward output:\n", output_ff)

Feedforward output:
 tensor([[[-0.0592, -0.0232, -0.2350, -0.0007],
         [-0.0720,  0.0135, -0.2071, -0.0657],
         [-0.0599, -0.0034, -0.2095, -0.0848]],

        [[-0.1396,  0.0120, -0.1898, -0.0588],
         [-0.0720,  0.0135, -0.2071, -0.0657],
         [-0.0599, -0.0034, -0.2095, -0.0848]],

        [[-0.1138,  0.0208, -0.1968, -0.0529],
         [-0.0720,  0.0135, -0.2071, -0.0657],
         [-0.0599, -0.0034, -0.2095, -0.0848]],

        [[-0.1138,  0.0208, -0.1968, -0.0529],
         [-0.0720,  0.0135, -0.2071, -0.0657],
         [-0.0599, -0.0034, -0.2095, -0.0848]],

        [[-0.1138,  0.0208, -0.1968, -0.0529],
         [-0.0720,  0.0135, -0.2071, -0.0657],
         [-0.0862,  0.0017, -0.2030, -0.0762]],

        [[-0.1138,  0.0208, -0.1968, -0.0529],
         [-0.0599, -0.0034, -0.2095, -0.0848],
         [-0.0599, -0.0034, -0.2095, -0.0848]],

        [[-0.1396,  0.0120, -0.1898, -0.0588],
         [-0.0720,  0.0135, -0.2071, -0.0657],
         [-0.0789,  0.0219,

In [35]:
class ResidualConnection(nn.Module):

    def __init__(self, dropout: float) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.norm = LayerNormalization()

    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x)))

In [36]:
class DecoderBlock(nn.Module):
    def __init__(self, self_attention_block: MultiHeadAttentionBlock, cross_attention_block: MultiHeadAttentionBlock, feed_forward_block: FeedForwardBlock, dropout : float) -> None:
        super().__init__()
        self.self_attention_block = self_attention_block
        self.cross_attention_block = cross_attention_block
        self.feed_forward_block = feed_forward_block
        self.residual_connections = nn.ModuleList([ResidualConnection(dropout) for _ in range(3)])

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, tgt_mask))
        x = self.residual_connections[1](x, lambda x: self.cross_attention_block(x, encoder_output, encoder_output, src_mask))
        x = self.residual_connections[2](x, self.feed_forward_block)
        return x




In [37]:
self_attention_block = MultiHeadAttentionBlock(d_model, h, dropout)
cross_attention_block = MultiHeadAttentionBlock(d_model, h, dropout)

decoder_block = DecoderBlock(
    self_attention_block=self_attention_block,
    cross_attention_block=cross_attention_block,
    feed_forward_block=feed_forward_block,
    dropout=dropout
)

src_mask = None  # Mask for source (encoder) input
tgt_mask = None  # Mask for target (decoder) input

decoder_output = decoder_block(output_ff, embeddings, src_mask, tgt_mask)

print("Decoder output:\n", decoder_output)


Decoder output:
 tensor([[[-2.7863,  1.1109, -0.6130,  0.1890],
         [-2.1589,  0.9483, -0.3529,  0.2118],
         [-2.0484,  1.5231, -0.5787, -0.1553]],

        [[-2.9230,  2.1514, -0.5320,  0.1795],
         [-2.9426,  2.2157, -0.6604, -0.0469],
         [-2.4942,  2.0679, -0.6377, -0.0098]],

        [[-2.3174,  1.8625, -0.6728,  0.1867],
         [-2.4386,  2.0625, -0.4217,  0.2869],
         [-2.7696,  2.0441, -0.6415, -0.0913]],

        [[-1.7456,  1.5838, -0.3440,  0.4927],
         [-2.8664,  2.1027, -0.6325,  0.4772],
         [-2.8737,  2.1390, -0.7273, -0.0982]],

        [[-1.5261,  1.8387, -0.4427,  0.1138],
         [-3.0014,  2.3187, -0.5747,  0.2621],
         [-2.3743,  1.8717, -0.3546,  0.1309]],

        [[-2.9464,  2.3448, -0.6195,  0.3091],
         [-2.2231,  1.8098, -0.4683, -0.0243],
         [-2.7616,  2.1491, -0.5989,  0.1013]],

        [[-3.1195,  2.3057, -0.5282,  0.2722],
         [-2.5030,  1.9444, -0.4343,  0.3391],
         [-2.5343,  1.9873,  0.